In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Online Retail II: Pembersihan Data

Notebook ini membersihkan data transaksi Online Retail II (UK, 2009-2011)
agar siap dipakai untuk analisis SQL, segmentasi RFM, dan cohort retensi.
Setiap langkah pembersihan dicatat jumlah baris yang dibuang beserta alasannya.

## 1. Memuat data
Data berisi 1.067.371 baris dan 8 kolom. Kolom `Customer ID` kosong pada
sekitar 22,8% baris dan terdapat 34.335 baris duplikat.

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv", encoding="ISO-8859-1")

print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())
print("Duplikat:", df.duplicated().sum())

## 2. Merapikan kolom dan tipe data
Nama kolom diseragamkan menjadi huruf kecil tanpa spasi, dan `InvoiceDate`
diubah dari teks menjadi tipe tanggal.

In [ ]:
df_raw = df.copy()  # simpan salinan asli
n_awal = len(df)

df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
df["invoice"] = df["invoice"].astype(str)

print("Rentang tanggal:", df["invoicedate"].min(), "sampai", df["invoicedate"].max())

## 3. Pembersihan bertahap
- **Duplikat** dihapus karena menggandakan transaksi yang sama.
- **Invoice berawalan "C"** adalah retur, dipisahkan ke tabel tersendiri.
- **Price <= 0 atau quantity <= 0** dihapus karena tidak mewakili penjualan valid.
- **Customer ID kosong** dihapus untuk analisis pelanggan (RFM dan cohort).

In [ ]:
log = []

def catat(langkah, sebelum, sesudah):
    log.append({"langkah": langkah, "dibuang": sebelum - sesudah, "sisa": sesudah})

# 1. Duplikat
n = len(df)
df = df.drop_duplicates()
catat("Hapus duplikat", n, len(df))

# 2. Pisahkan retur (invoice berawalan C) dan simpan terpisah
returns = df[df["invoice"].str.startswith("C")].copy()
n = len(df)
df = df[~df["invoice"].str.startswith("C")]
catat("Pisahkan transaksi retur (C)", n, len(df))

# 3. Harga dan kuantitas tidak valid
n = len(df)
df = df[(df["price"] > 0) & (df["quantity"] > 0)]
catat("Hapus price <= 0 atau quantity <= 0", n, len(df))

# 4. Customer ID kosong (untuk analisis pelanggan)
n = len(df)
df_pelanggan = df.dropna(subset=["customer_id"]).copy()
df_pelanggan["customer_id"] = df_pelanggan["customer_id"].astype(int)
catat("Hapus customer_id kosong", n, len(df_pelanggan))

# 5. Kolom turunan
df_pelanggan["total_price"] = df_pelanggan["quantity"] * df_pelanggan["price"]
df_pelanggan["invoice_month"] = df_pelanggan["invoicedate"].dt.to_period("M").astype(str)

pd.DataFrame(log)

In [ ]:
print(df_pelanggan.shape)
print(df_pelanggan[["quantity", "price", "total_price"]].describe())

df.to_csv("/kaggle/working/sales_all.csv", index=False)
df_pelanggan.to_csv("/kaggle/working/sales_customers.csv", index=False)
returns.to_csv("/kaggle/working/returns.csv", index=False)

## 4. Menelusuri outlier dan baris non-produk
Kode seperti POST (ongkos kirim), M (input manual), BANK CHARGES, ADJUST,
dan TEST bukan penjualan produk, sehingga dikeluarkan dari analisis produk.
Tiga transaksi terbesar (80.995, 74.215, dan 60 unit) terbukti dibatalkan:
tiap invoice punya pasangan retur berawalan "C" dengan kuantitas yang sama.
Pesanan grosir besar lainnya tetap dipertahankan karena sah.

In [ ]:
# 1. Kode stok yang bukan produk (ongkir, biaya, penyesuaian)
non_produk = df_pelanggan[~df_pelanggan["stockcode"].astype(str).str.match(r"^\d{5}")]
print(non_produk["stockcode"].value_counts().head(15))
print("Total pendapatan non-produk:", non_produk["total_price"].sum())

# 2. 10 transaksi terbesar
print(df_pelanggan.nlargest(10, "total_price")[
    ["invoice","stockcode","description","quantity","price","total_price","customer_id"]])

# 3. Persentil atas
print(df_pelanggan[["quantity","total_price"]].quantile([0.95, 0.99, 0.999]))

In [ ]:
besar = [(16446, 80995), (12346, 74215), (15098, 60)]
for cid, qty in besar:
    cek = returns[(returns["customer_id"] == cid) & (returns["quantity"].abs() == qty)]
    print(f"Pelanggan {cid}, qty {qty}:")
    print(cek[["invoice", "invoicedate", "stockcode", "quantity", "price"]], "\n")

## 5. Data akhir
Setelah semua langkah, tersisa 776.574 baris (72,8% dari data awal).
Total pendapatan turun dari 17,37 juta menjadi 16,78 juta.
Data ini disimpan sebagai `sales_products_clean.csv`.

In [ ]:
kode_nonproduk = ["POST","M","C2","ADJUST","ADJUST2","BANK CHARGES",
                  "PADS","DOT","D","SP1002","TEST001","TEST002"]
invoice_batal = ["581483", "541431", "556444"]

n = len(df_pelanggan)
df_produk = df_pelanggan[
    ~df_pelanggan["stockcode"].astype(str).isin(kode_nonproduk)
    & ~df_pelanggan["invoice"].isin(invoice_batal)
].copy()
catat("Hapus kode non-produk dan 3 invoice yang dibatalkan", n, len(df_produk))

print(pd.DataFrame(log))
print("Pendapatan sebelum:", round(df_pelanggan["total_price"].sum()))
print("Pendapatan sesudah:", round(df_produk["total_price"].sum()))

df_produk.to_csv("/kaggle/working/sales_products_clean.csv", index=False)

## Keterbatasan
Sekitar 21% baris tidak memiliki Customer ID sehingga tidak masuk analisis
pelanggan. Hasil RFM dan cohort hanya mewakili transaksi yang tercatat
atas nama pelanggan.